# 2-motor: teardown

Open the motor last, read the copper's resistance against its own temperature with no brushes in the path, and weigh and count every part, so the winding thermometer and the thermal model of stage 3 rest on the copper itself and not on a terminal reading.

## Purpose

Every electrical reading of the motor so far went in through the brushes. The terminal resistance of rest-resistance is the copper plus two brush contacts and the commutator, and the thermal experiments of stage 3 read the copper's temperature from how that resistance rises. The rise it is looking for is the copper's temperature coefficient, which for pure copper is 0.00393 per C referenced to 20 C. A winding wound from the usual enameled wire should read close to that. A winding that reads something else, or a brush path that is a large share of the terminal number and moves on its own, is what the thermometer has to live with. This experiment reads the copper directly. The rotor is out of its can, two leads are soldered to two of the commutator tabs, a thermocouple sits in a coil, and the rotor goes through four temperature plateaus on the heater. The line through the plateaus gives the coefficient and the resistance at 20 C. With the rest-resistance result, the difference between the terminal resistance and the copper at the same temperature is the brush path.

The same teardown gives the things that can only be read with the motor open. The masses of the rotor, the can with its magnets and the end cap feed the thermal model, since the rotor's mass is the heat capacity the copper warms, and the can's is the heat sink it warms into. The counts of slots, commutator segments, brushes and magnet arcs pin the commutation ripple per revolution that bemf-coast counts the speed with, and the registry entry carries the counts afterwards. The photos are the record. This is done once for the MG90 clone family, as the principles say, and the numbers score the non-destructive methods on the next motor of the family rather than being measured again.

## Procedure

The motor comes apart after every other stage 2 experiment has run on it. The can's crimp tabs are bent back, the end cap with the brush card comes off, and the rotor comes out with its three coils and the commutator. Two leads are soldered to two of the three commutator tabs, the small tabs at the base of the commutator where the coil wires are hooked and soldered, and not to the running surface the brushes ride on. The thermocouple tip is pressed into the gap between two coils, against the copper, and taped over with kapton so it stays. The rotor then lies on the heated bed of a 3D printer, the same bed as the stage 1 thermal pair, with the shaft on the bed and the coils up.

The meter has one input, so at each plateau it reads ohms for a minute and then the thermocouple for a minute, the dial turned by hand at the prompt. The heater runs at two settings and then off, like ntc-curve.

1. The two leads shorted together at their tips, the DMM on ohms, Bluetooth on. `capture.py` reads the lead zero for thirty seconds first.
2. The leads on the commutator tabs, the thermocouple in a coil, the rotor on the heater, the heater off. The script logs ohms for a minute at the `room ohm` mark, then asks for the dial on temperature and logs the thermocouple for a minute at the `room C` mark.
3. The heater set to about 45 C. Once the thermocouple has settled, the same two minutes at the `warm` marks.
4. The heater set to about 60 C, the same at the `hot` marks.
5. The heater off, the rotor left to come back near room, the same at the `room again` marks, and the `end` mark closes the run.

The 60 C ceiling keeps the enamel, the solder on the tabs and the kapton well inside what they take. The bed's own thermistor sits in the heater and is not a temperature reference here, as ntc-curve found.

## Data

Ten files in the capture folder, all on the host clock, each a meter log with the same columns, `t_host`, `function`, `value` and `raw` the packet.

- `zero.csv`: the leads shorted, `function` is `ohm` and `value` in ohm.
- `<plateau>-ohm.csv`: the ohms window of each plateau, `room`, `warm`, `hot` and `room-again`.
- `<plateau>-c.csv`: the thermocouple window of each plateau, `function` is `C` and `value` in C as the meter displays it.
- `marks.csv`: the host time of `lead zero`, then `<plateau> ohm` and `<plateau> C` for each plateau, then `end`. A mark names the window that starts at it.

### The logs

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, laps, motors, results
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"   # the house colors
MOTOR = motors.MOTORS[motors.DEFAULT]                                    # the motor, with the counts the registry holds
CAPTURE = (Path("../../../telemetry") / MOTOR.key / "teardown" / "capture-1").resolve()   # no board in a teardown's path
PLATEAUS = ["room", "warm", "hot", "room again"]                        # the four plateaus in the order the heater ran them
ALPHA_COPPER = 0.00393                                                  # pure copper's temperature coefficient per C, referenced to 20 C
T_REF = 20.0                                                            # the reference temperature of R20 and alpha, in C
print(MOTOR.key, "from", results.rel(CAPTURE))
print(f"{MOTOR.label}: {MOTOR.slots} slots, {MOTOR.brushes} brushes, {MOTOR.pinion_teeth} pinion teeth, as the registry counts them")

In [ ]:
def log(name, function):
    # one meter log as a frame, kept to the packets on the function the window was logged on
    df = b41t.load(CAPTURE / f"{name}.csv")
    return df[df["function"] == function].dropna(subset=["value"]).reset_index(drop=True)

slug = lambda p: p.replace(" ", "-")                                       # "room again" is room-again on disk
zero = log("zero", "ohm")                                                  # the leads shorted, on ohms
ohm = {p: log(f"{slug(p)}-ohm", "ohm") for p in PLATEAUS}                  # one ohms window per plateau
tc = {p: log(f"{slug(p)}-c", "C") for p in PLATEAUS}                       # one thermocouple window per plateau
marks = pd.read_csv(CAPTURE / "marks.csv")                                 # the host time of each prompt
mt = marks.set_index("mark")["t_host"]                                     # the marks by name
t0 = mt["lead zero"]                                                       # time zero for the run
TC = results.read("0-instruments", "thermocouple")                         # the thermocouple's offset and slope against ice and boiling water
DMM = results.read("0-instruments", "dmm")                                 # the meter's classes from its manual, ohms and temperature
for p in PLATEAUS:
    tc[p]["t_c"] = (tc[p]["value"] - TC["offset_c"]["value"]) / TC["slope"]["value"]   # the stage 0 correction: t = (reading - offset) / slope
OHM_PCT, OHM_FIXED = DMM["accuracy"]["ohm_220"]["pct"]["value"], DMM["accuracy"]["ohm_220"]["fixed"]["value"]
C_PCT, C_FIXED = DMM["accuracy"]["temp_c"]["pct"]["value"], DMM["accuracy"]["temp_c"]["fixed"]["value"]
ohm_class = lambda r: OHM_FIXED + OHM_PCT / 100 * np.asarray(r, float)    # the meter's ohms class at a reading, in ohm
c_class = lambda t: C_FIXED + C_PCT / 100 * np.asarray(t, float)          # the meter's temperature class at a reading, in C
show(zero); show(ohm["room"]); show(tc["room"])
show(marks.assign(minutes=(marks["t_host"] - t0) / 60), n=None)
print(f"lead zero: {len(zero)} packets over {zero['t'].iloc[-1]:.0f} s, range byte {zero['raw'].astype(str).str[:4].unique().tolist()}")
for p in PLATEAUS:
    print(f"{p:11s} ohms {len(ohm[p]):3d} packets over {ohm[p]['t'].iloc[-1]:.0f} s, range byte {ohm[p]['raw'].astype(str).str[:4].unique().tolist()}; "
          f"thermocouple {len(tc[p]):3d} packets over {tc[p]['t'].iloc[-1]:.0f} s, longest gap {np.diff(tc[p]['t_host']).max():.2f} s")
print(f"thermocouple correction: offset {TC['offset_c']['value']:.1f} C, slope {TC['slope']['value']:.4f}; "
      f"meter class {OHM_PCT:g} % + {OHM_FIXED:g} ohm on the 220 ohm range, {C_PCT:g} % + {C_FIXED:g} C on temperature")

The meter reads about twice a second on both functions, ohms in 0.01 ohm steps on its 220 ohm range and the thermocouple in 0.1 C steps. The range byte of every ohms packet is the same when the meter stayed on one range through the run, which the medians below assume. The marks are listed in minutes from the lead zero mark, which is time zero for the figures.

### The run

The whole run on one time axis, the ohms windows on the top panel with the lead zero first, the corrected thermocouple windows on the bottom, each window shaded and each mark named. The thing to see is whether the resistance and the temperature are still moving inside a window.

In [ ]:
tm = lambda t: (np.asarray(t, float) - t0) / 60                           # host seconds to minutes from the lead zero mark
fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
ax[0].plot(tm(zero["t_host"]), zero["value"], ".", ms=2, color=MUTED, label="the leads shorted")
for p in PLATEAUS:
    ax[0].plot(tm(ohm[p]["t_host"]), ohm[p]["value"], ".", ms=2, color=BLUE)
    ax[1].plot(tm(tc[p]["t_host"]), tc[p]["t_c"], ".", ms=2, color=ORANGE)
    for a, w in ((ax[0], ohm[p]), (ax[1], tc[p])):                            # the window each log covers
        a.axvspan(tm(w["t_host"].iloc[0]), tm(w["t_host"].iloc[-1]), color=MUTED, alpha=0.15)
for a in ax:
    for _, m in marks.iterrows():                                          # the marks
        a.axvline(tm(m["t_host"]), color=MUTED, lw=0.6)
    a.grid(alpha=0.3)
for i, (_, m) in enumerate(marks.iterrows()):                              # the mark names on two rows in turn, so neighbors do not collide
    end = m["mark"] == "end"
    ax[0].text(tm(m["t_host"]) + (-0.1 if end else 0.1), 0.97 if i % 2 == 0 else 0.90, m["mark"], fontsize=8, va="top",
               ha="right" if end else "left", transform=ax[0].get_xaxis_transform())
ax[0].set_ylabel("meter (ohm), leads included"); ax[0].legend(fontsize=8, loc="lower right")
ax[0].set_title("the run: ohms and the thermocouple by turns at each plateau")
ax[1].set_ylabel("thermocouple, corrected (C)"); ax[1].set_xlabel("minutes from the lead zero mark")
plt.tight_layout(); plt.show()

Each plateau is a pair of windows a minute apart, the ohms first and the thermocouple after it, so the two readings of a plateau are not taken together and the plateau has to be flat for the pair to belong to one temperature. The creep inside each window, in the table below, is the check.

## Method

The lead zero is the median of the shorted-leads log, and every ohms reading has it taken off. A plateau's resistance is the median of its ohms window less the lead zero, with the window's standard deviation as its spread, and its temperature is the median of the corrected thermocouple window, the meter's reading put through the stage 0 offset and slope. The creep of each window, the second half's median less the first half's, says whether the rotor was still moving while it was read. A window with a creep well over its spread is a point on a ramp and not a plateau, and the room-again plateau is the one most likely to show it, since a rotor cooling on a bed that is itself cooling comes to room slowly.

The model is the linear temperature coefficient referenced to 20 C,

$$R = R_{20} \, (1 + \alpha \, (T - 20))$$

with $R$ the resistance between the two tabs in ohm, $T$ the copper's temperature in C from the thermocouple, $R_{20}$ the resistance at 20 C and $\alpha$ the coefficient per C. A line $R = a + b\,T$ is fitted by least squares through the four plateaus, then $R_{20} = a + 20\,b$ and $\alpha = b / R_{20}$, with the slope's standard error carried into $\alpha$. The line is fitted again over the row choices, the three plateaus of the heat-up alone, the two room plateaus against the hot one, and the whole set, and the spread over those choices is the bracket the result file carries on $\alpha$. The coefficient is compared with pure copper's 0.00393 per C. A fit near it says the method reads the copper. A fit well under it would point at something in series that does not warm with the copper, the solder joints and the tabs, or at a thermocouple that is not at the copper's temperature.

The meter bounds the absolute numbers. Its ohms class on the 220 ohm range is 0.5 % of the reading plus 0.1 ohm, from the stage 0 dmm result. A scale error of the meter cancels in $\alpha$, since $\alpha$ is a slope over a resistance read on the same scale. An offset does not cancel, it moves $R_{20}$ and so $\alpha$ by the offset over $R_{20}$, and the 0.1 ohm fixed part is that offset's bound once the lead zero is out. The temperature class is 1 % of the reading plus 0.5 C, and in a slope over a 40 C span the fixed part cancels and the 1 % stays.

The reading between two tabs is not one coil. A three-slot rotor has three coils, and the usual winding joins them in a ring, each coil between two neighboring commutator segments, so between any two tabs one coil sits in parallel with the other two in series. The reading is then $R_c \parallel 2 R_c = \tfrac{2}{3} R_c$ and one coil is $R_c = 1.5 \times$ the reading. The brushes contact two segments as well, so the winding as the brushes see it between two commutator points is the same $\tfrac{2}{3} R_c$, which is the tab reading itself, and at the angles where one brush spans the gap between two segments it shorts one coil and the other two sit in parallel, $R_c / 2$, three quarters of the tab reading. That ring connection is the assumption, and the teardown's photos of the commutator show which coil end goes to which tab and confirm it. A star connection, the three coil ends joined at a center, would read $2 R_c$ between tabs instead and the division would differ.

The brush path comes last. The rest-resistance result carries the terminal resistance at rest by the meter's route, `r_ohms_median_ohm` with its bounds, the slope of the stall laps' volts against amps, `r_stall_ohm`, and the brush drop their intercept gives, `v0_brush_v`. The copper at that sitting's temperature is $R_{20}$ of the winding put through the measured $\alpha$, and the brush path is the terminal resistance less it, once by the meter's route, which carries the brushes at the meter's milliamp, and once by the stall route, which carries the contact's ohmic part at hundreds of milliamps with the drop taken out as $V_0$. The rest-resistance result's own room temperature is used when it carries one, else this run's room plateau, and the result notes which.

### The plateaus

The median, spread and creep of each window, the resistance with the lead zero out, and the meter's class at each reading.

In [ ]:
LEAD_ZERO = zero["value"].median()                                         # the leads' own resistance, taken off every reading
creep = lambda s: s.iloc[len(s) // 2:].median() - s.iloc[:len(s) // 2].median()   # second half less first: the movement inside a window
rows = []
for p in PLATEAUS:
    r, t = ohm[p]["value"], tc[p]["t_c"]
    rows.append({"plateau": p, "ohm_n": len(r), "r_raw_ohm": r.median(), "r_sd": r.std(), "r_creep": creep(r),
                 "c_n": len(t), "t_c": t.median(), "t_sd": t.std(), "t_creep": creep(t),
                 "gap_s": tc[p]["t_host"].iloc[0] - ohm[p]["t_host"].iloc[-1]})   # from the last ohms packet to the first thermocouple packet
P = pd.DataFrame(rows).set_index("plateau")
P["r_ohm"] = P["r_raw_ohm"] - LEAD_ZERO                                    # the tabs alone
P["ohm_class"] = ohm_class(P["r_ohm"]); P["c_class"] = c_class(P["t_c"])
show(P, n=None)
print(f"lead zero {LEAD_ZERO:.3f} ohm, sd {zero['value'].std():.3f}, {len(zero)} packets")
print(f"the thermocouple's correction adds {P['t_c'].iloc[0] - tc['room']['value'].median():+.2f} C at room and "
      f"{P['t_c'].loc['hot'] - tc['hot']['value'].median():+.2f} C at the hot plateau")
print("creep inside the windows, second half less first: "
      + ", ".join(f"{p} {P.loc[p, 'r_creep']:+.3f} ohm / {P.loc[p, 't_creep']:+.2f} C" for p in PLATEAUS))

One row per plateau. The resistance's spread over a window is the meter's step or a little over it when the rotor sits still, and the thermocouple's is its 0.1 C step. A creep over the spread marks a window read on a ramp.

## Results

### The line through the plateaus

The resistance between the tabs against the corrected thermocouple, one point per plateau with its spread as the whiskers, the fitted line, and pure copper's slope drawn through the same $R_{20}$ beside it. The residuals from the line sit under it. The thing to see is whether the four points lie on one line and how far its slope sits from copper's.

In [ ]:
fit = laps.line(P["t_c"], P["r_ohm"])                                        # R = a + b T through the four plateaus
r20_of = lambda f: f["intercept"] + f["slope"] * T_REF                        # R20 from a line
alpha_of = lambda f: f["slope"] / r20_of(f)                                   # alpha referenced to 20 C
R20_TABS, ALPHA = r20_of(fit), alpha_of(fit)
ALPHA_SE = fit["slope_se"] / R20_TABS                                         # the slope's standard error carried into alpha
choices = {"all four plateaus": fit,
           "heat-up alone (room, warm, hot)": laps.line(P.loc[["room", "warm", "hot"], "t_c"], P.loc[["room", "warm", "hot"], "r_ohm"]),
           "the two rooms and the hot plateau": laps.line(P.loc[["room", "hot", "room again"], "t_c"], P.loc[["room", "hot", "room again"], "r_ohm"])}
alphas = [alpha_of(f) for f in choices.values()]
ALPHA_LO, ALPHA_HI = min(alphas + [ALPHA - ALPHA_SE]), max(alphas + [ALPHA + ALPHA_SE])   # the bracket: the row choices and the fit's own error
R20_CLASS = ohm_class(R20_TABS)                                               # the meter's class at R20, the bound on the absolute number
P["r_fit"] = fit["intercept"] + fit["slope"] * P["t_c"]
P["resid"] = P["r_ohm"] - P["r_fit"]
print(f"R = {fit['intercept']:.4f} + {fit['slope']:.5f} +/- {fit['slope_se']:.5f} ohm/C x T, rms {fit['rms']:.4f} ohm over {fit['n']} plateaus")
print(f"R20 between the tabs {R20_TABS:.4f} ohm (the meter's class there {R20_CLASS:.3f} ohm); "
      f"alpha {ALPHA:.5f} +/- {ALPHA_SE:.5f} per C, {ALPHA / ALPHA_COPPER * 100:.1f} % of copper's {ALPHA_COPPER}")
print("the row choice as a check")
for name, f in choices.items():
    print(f"  {name:36s} R20 {r20_of(f):.4f} ohm, alpha {alpha_of(f):.5f} per C ({alpha_of(f) / ALPHA_COPPER * 100:.1f} % of copper)")
print(f"bracket on alpha {ALPHA_LO:.5f} to {ALPHA_HI:.5f} per C; an offset of {OHM_FIXED:g} ohm in the meter would move alpha by "
      f"{OHM_FIXED / R20_TABS * 100:.1f} %, the thermocouple's {C_PCT:g} % class by {C_PCT:g} % of the slope")
print(f"room again against room: {P.loc['room again', 'r_ohm'] - P.loc['room', 'r_ohm']:+.4f} ohm at {P.loc['room again', 't_c'] - P.loc['room', 't_c']:+.2f} C, "
      f"the line predicts {fit['slope'] * (P.loc['room again', 't_c'] - P.loc['room', 't_c']):+.4f} ohm")

fig, ax = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True, gridspec_kw={"height_ratios": [2.4, 1]})
tt = np.linspace(P["t_c"].min() - 3, P["t_c"].max() + 3, 20)
ax[0].errorbar(P["t_c"], P["r_ohm"], xerr=P["t_sd"], yerr=P["r_sd"], fmt="o", ms=7, color=BLUE, mec=INK, capsize=3, label="plateau medians with their spread")
for p in PLATEAUS:
    ax[0].annotate(p, (P.loc[p, "t_c"], P.loc[p, "r_ohm"]), xytext=(8, -12), textcoords="offset points", fontsize=9, color=INK)
ax[0].plot(tt, fit["intercept"] + fit["slope"] * tt, "-", color=INK, lw=1.2, label=f"fit: R20 {R20_TABS:.3f} ohm, alpha {ALPHA:.5f} per C")
ax[0].plot(tt, R20_TABS * (1 + ALPHA_COPPER * (tt - T_REF)), ls="dashed", color=MUTED, lw=1.2, label=f"pure copper, {ALPHA_COPPER} per C through the same R20")
ax[0].set_ylabel("between the tabs, lead zero out (ohm)"); ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)
ax[0].set_title("the copper's resistance against its temperature, one point per plateau")
ax[1].errorbar(P["t_c"], P["resid"], yerr=P["r_sd"], fmt="o", ms=6, color=BLUE, mec=INK, capsize=3)
ax[1].axhline(0, color=INK, lw=0.5); ax[1].set_ylabel("residual from the fit (ohm)"); ax[1].set_xlabel("thermocouple, corrected (C)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

The points, the line and the copper line, with the residuals under them. A slope near copper's with residuals inside the spread says the thermocouple sat at the copper's temperature at every plateau and the reading is the copper. The room-again point against the room point is the hysteresis check, two readings of the same copper at nearly the same temperature on either side of the heating.

### The winding and the brush path

The tab reading split into one coil and the winding as the brushes see it under the ring connection, then the terminal resistance from rest-resistance at that sitting's temperature with the copper taken out, by the meter's route and by the stall route. The thing to see in the figure is how much of the terminal resistance is copper and how much is the brush path, on each route.

In [ ]:
R20_COIL = 1.5 * R20_TABS                                                     # one coil: the tabs read one coil in parallel with the other two in series
R20_WINDING = R20_TABS                                                        # the brushes on two segments see the same 2/3 of a coil as the tabs
R20_WINDING_BRIDGE = R20_COIL / 2                                             # a brush spanning a gap shorts one coil, the other two in parallel
R20_LO, R20_HI = min(r20_of(f) for f in choices.values()), max(r20_of(f) for f in choices.values())   # R20 over the row choices
r_at = lambda r20, t: r20 * (1 + ALPHA * (t - T_REF))                         # the copper at another temperature through the measured alpha
RR = results.read("2-motor", "rest-resistance", pair=MOTOR.key)               # the terminal resistance at rest, the stall slope and the brush drop
T_RR_KEY = next((k for k in ("room_c", "t_c", "temperature_c") if k in RR), None)   # that sitting's room, when the result carries it
T_RR = RR[T_RR_KEY]["value"] if T_RR_KEY else P.loc["room", "t_c"]
T_RR_FROM = f"the rest-resistance result's {T_RR_KEY}" if T_RR_KEY else "this run's room plateau, the rest-resistance result carrying no temperature"
R_TERM, R_STALL, V0 = RR["r_ohms_median_ohm"], RR["r_stall_ohm"], RR["v0_brush_v"]
R_WINDING_RR = r_at(R20_WINDING, T_RR)                                        # the copper as the brushes saw it at that sitting
BRUSH_METER = R_TERM["value"] - R_WINDING_RR                                  # the brush path at the meter's milliamp
BRUSH_METER_LO = (R_TERM["lo"] if R_TERM["lo"] is not None else R_TERM["value"]) - r_at(R20_HI, T_RR)
BRUSH_METER_HI = (R_TERM["hi"] if R_TERM["hi"] is not None else R_TERM["value"]) - r_at(R20_LO, T_RR)
BRUSH_STALL = R_STALL["value"] - R_WINDING_RR                                 # the contact's ohmic part at the stall laps' current, the drop apart as V0
print(f"R20 between the tabs {R20_TABS:.4f} ohm ({R20_LO:.4f} to {R20_HI:.4f} over the row choices): one coil {R20_COIL:.4f} ohm, "
      f"the winding as the brushes see it {R20_WINDING:.4f} ohm on two segments and {R20_WINDING_BRIDGE:.4f} ohm with a brush across a gap")
print(f"rest-resistance sat at {T_RR:.1f} C ({T_RR_FROM}): "
      f"the copper there {R_WINDING_RR:.4f} ohm")
print(f"terminal at rest by the meter {R_TERM['value']:.4f} ohm: brush path {BRUSH_METER:.4f} ohm ({BRUSH_METER_LO:.4f} to {BRUSH_METER_HI:.4f}), "
      f"{BRUSH_METER / R_TERM['value'] * 100:.0f} % of the terminal reading")
print(f"stall slope {R_STALL['value']:.4f} ohm with V0 {V0['value']:.3f} V: the contact's ohmic part {BRUSH_STALL:.4f} ohm, {BRUSH_STALL / R_STALL['value'] * 100:.0f} % of the slope")
print(f"the ring connection predicts the terminal reading drops to {R20_WINDING_BRIDGE / R20_WINDING * 100:.0f} % of itself at the bridging angles, "
      f"less the brush path, which the twelve holes of rest-resistance can be read against")

fig, ax = plt.subplots(figsize=(8, 3.8))
routes = {"meter at rest": (R_WINDING_RR, BRUSH_METER), "stall slope": (R_WINDING_RR, BRUSH_STALL)}
y = np.arange(len(routes))
ax.barh(y, [c for c, _ in routes.values()], color=ORANGE, label="copper, the winding at that sitting's temperature")
ax.barh(y, [b for _, b in routes.values()], left=[c for c, _ in routes.values()], color=BLUE, label="brush path, the terminal reading less the copper")
ax.set_yticks(y, list(routes)); ax.set_xlabel("ohm"); ax.invert_yaxis()
ax.set_title("where the terminal resistance sits: copper against the brush path, by route"); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

The copper's share and the brush path's share of the terminal resistance, by the meter's route at a milliamp and by the stall route at hundreds of milliamps. The two routes read the brushes under very different currents and are not expected to agree, since a brush contact is not ohmic. The gap between them, with the drop $V_0$ beside it, is what the in-situ thermometer of stage 3 has to carry as the part of the terminal number that is not copper.

## Teardown

The hands part. Everything in this section is read off the opened motor and typed in by hand before the notebook runs. The photos go in a `photos/` folder beside this notebook, and these four are the ones to take. The open can with the magnets in it, the rotor on its own, the commutator with the leads soldered on so the coil ends can be followed to the tabs, and the brush card with its two brushes. A scale or a coin in the rotor photo gives it a size.

### Masses

A kitchen scale with 0.1 g steps, or better. The whole motor is weighed before the teardown, each part after it, and the parts have to add up to the whole to within the scale's step and the solder on the tabs.

| Part | Mass (g) |
| - | - |
| whole motor, before the teardown | to fill |
| rotor, with the commutator and the shaft | to fill |
| can with its magnets | to fill |
| end cap with the brush card | to fill |
| pinion, when it comes off the shaft | to fill |

### Counts

| Count | Registry | Counted |
| - | - | - |
| rotor slots | from the registry | to fill |
| commutator segments | | to fill |
| brushes | from the registry | to fill |
| pinion teeth | from the registry | to fill |
| magnet arcs | | to fill |

### Dimensions

Calipers, in millimeters. The rotor's diameter and stack length, the shaft's diameter, each magnet arc's length and thickness, and the can's outer diameter and length. The rotor's dimensions and mass together say how much of it is iron and how much copper, and the magnet arcs' dimensions go with the back-EMF constant.

### Photos

![the open can with its magnets](photos/open-can.jpg)

![the rotor, three coils and the commutator](photos/rotor.jpg)

![the commutator with the two leads on its tabs](photos/commutator-leads.jpg)

![the brush card in the end cap](photos/brush-card.jpg)

Below, the same numbers as code so they land in the result file. A value left at `None` stays out of the file and is named in the output.

In [ ]:
MASSES_G = {"whole motor": None, "rotor": None, "can with magnets": None, "end cap": None, "pinion": None}   # grams, by hand
COUNTS = {"slots": None, "commutator segments": None, "brushes": None, "pinion teeth": None, "magnet arcs": None}   # as counted on the open motor
DIMENSIONS_MM = {"rotor diameter": None, "rotor stack length": None, "shaft diameter": None,
                 "magnet arc length": None, "magnet thickness": None, "can outer diameter": None, "can length": None}
REGISTRY_COUNTS = {"slots": MOTOR.slots, "brushes": MOTOR.brushes, "pinion teeth": MOTOR.pinion_teeth}   # what the registry says
unfilled = [k for d in (MASSES_G, COUNTS, DIMENSIONS_MM) for k, v in d.items() if v is None]
print("not yet filled in:", ", ".join(unfilled) if unfilled else "nothing, every value is in")
parts = [MASSES_G[k] for k in ("rotor", "can with magnets", "end cap") if MASSES_G[k] is not None]
if len(parts) == 3 and MASSES_G["whole motor"] is not None:
    print(f"the parts add up to {sum(parts):.1f} g against {MASSES_G['whole motor']:.1f} g whole, {sum(parts) - MASSES_G['whole motor']:+.1f} g")
for k, n in REGISTRY_COUNTS.items():
    if COUNTS[k] is not None:
        print(f"{k}: the registry counts {n}, the teardown counted {COUNTS[k]}" + ("" if COUNTS[k] == n else " (the registry entry is wrong and gets the counted value)"))
if COUNTS["slots"] is not None and COUNTS["commutator segments"] is not None and COUNTS["slots"] != COUNTS["commutator segments"]:
    print("the segment count differs from the slot count, so the ring connection assumed above does not hold as written")

What was typed in, and whether the parts add up to the whole and the counts match the registry.

### The results file

The coefficient with its bracket, the resistance at 20 C between the tabs, of one coil and of the winding as the brushes see it, the brush path by both routes, the lead zero, each plateau's resistance and temperature with their spreads, and the masses, counts and dimensions as typed. The stage 3 thermal experiments read `alpha_per_c` and `r20_winding_ohm`, and the registry entry takes the counts.

In [ ]:
q = results.quantity
plateaus = {slug(p): {"r_ohm": q(P.loc[p, "r_ohm"], P.loc[p, "r_ohm"] - P.loc[p, "r_sd"], P.loc[p, "r_ohm"] + P.loc[p, "r_sd"], "ohm"),
                      "t_c": q(P.loc[p, "t_c"], P.loc[p, "t_c"] - P.loc[p, "t_sd"], P.loc[p, "t_c"] + P.loc[p, "t_sd"], "C"),
                      "r_creep_ohm": q(P.loc[p, "r_creep"], unit="ohm"), "t_creep_c": q(P.loc[p, "t_creep"], unit="C"),
                      "ohm_class": q(P.loc[p, "ohm_class"], unit="ohm"), "c_class": q(P.loc[p, "c_class"], unit="C")} for p in PLATEAUS}
by_hand = lambda d, unit: {k.replace(" ", "_"): q(v, unit=unit) for k, v in d.items() if v is not None}
results.write("2-motor", "teardown", CAPTURE, pair=MOTOR.key,
              notes="the rotor out of its can on the printer bed, two leads on two commutator tabs, the thermocouple in a coil; r_ohm per plateau is the "
                    "median of the ohms window less lead_zero_ohm, t_c the median of the thermocouple window after its stage 0 correction, lo/hi one sd of "
                    "the window; R = R20 (1 + alpha (T - 20)) through the four plateaus, alpha's lo/hi the spread over the row choices and the fit's "
                    "standard error; r20_coil_ohm = 1.5 x the tab reading and r20_winding_ohm = the tab reading under the ring connection, one coil in "
                    "parallel with the other two in series, which the photos confirm; r20_winding_bridging_ohm is the brushes' view with one brush across "
                    "a gap; brush_path_ohm is rest-resistance's r_ohms_median_ohm less the winding at that sitting's temperature through alpha, "
                    f"brush_path_stall_ohm the same from r_stall_ohm, both at {T_RR:.1f} C from {T_RR_FROM}"
                    "; masses_g, counts and dimensions_mm are typed in by hand from the opened motor",
              alpha_per_c=q(ALPHA, lo=ALPHA_LO, hi=ALPHA_HI, unit="1/C"),
              alpha_copper_per_c=q(ALPHA_COPPER, unit="1/C"),
              alpha_over_copper_pct=q(ALPHA / ALPHA_COPPER * 100, unit="%"),
              r20_between_tabs_ohm=q(R20_TABS, lo=R20_LO, hi=R20_HI, unit="ohm"),
              r20_coil_ohm=q(R20_COIL, lo=1.5 * R20_LO, hi=1.5 * R20_HI, unit="ohm"),
              r20_winding_ohm=q(R20_WINDING, lo=R20_LO, hi=R20_HI, unit="ohm"),
              r20_winding_bridging_ohm=q(R20_WINDING_BRIDGE, lo=0.75 * R20_LO, hi=0.75 * R20_HI, unit="ohm"),
              r20_meter_class_ohm=q(R20_CLASS, unit="ohm"),
              brush_path_ohm=q(BRUSH_METER, lo=BRUSH_METER_LO, hi=BRUSH_METER_HI, unit="ohm"),
              brush_path_stall_ohm=q(BRUSH_STALL, unit="ohm"),
              brush_path_at_c=q(T_RR, unit="C"),
              lead_zero_ohm=q(LEAD_ZERO, lo=LEAD_ZERO - zero["value"].std(), hi=LEAD_ZERO + zero["value"].std(), unit="ohm"),
              fit_rms_ohm=q(fit["rms"], unit="ohm"),
              range_c=q(P["t_c"].max(), lo=P["t_c"].min(), hi=P["t_c"].max(), unit="C"),
              plateaus=plateaus,
              masses_g=by_hand(MASSES_G, "g"),
              counts=by_hand(COUNTS, "count"),
              dimensions_mm=by_hand(DIMENSIONS_MM, "mm"))

## Limitations

- The meter bounds the absolute resistance. Its ohms class on the 220 ohm range is 0.5 % of the reading plus 0.1 ohm, and on a winding of a few ohm the fixed part is the larger share, so $R_{20}$ is known to about that through this meter. The coefficient is a slope over that same reading, so the meter's scale cancels in it and only an offset of up to 0.1 ohm moves it, by that offset over $R_{20}$.
- The ohms and the thermocouple are read a minute apart at each plateau, since the meter has one input, so the pair belongs to one temperature only where the plateau is flat, and the creep inside each window is the only check.
- The thermocouple tip is pressed into a coil under kapton with no paste. The ntc-curve pad test showed a taped tip in still air reads under its surface by a share of the surface's excess over the room, and a tip inside a coil is better shielded than one on a flat board, but how much it reads under the copper is not measured here. A tip reading low makes the slope read high and the coefficient with it.
- Two tabs of three. The coefficient does not depend on which pair, but $R_{20}$ between the tabs and the coil's value through it rest on the ring connection, which the photos confirm and the numbers do not test on their own.
- Four plateaus over room to about 60 C. The line is fitted through four points, so the residuals say little about curvature, and copper is linear enough over that span for the question not to matter.
- The rotor is read on the bed with the shaft down, so the coils warm from the iron up and the heat reaches the copper through the stack. The windows are placed where the thermocouple has settled, and a plateau that was still creeping is a point on a ramp.
- One motor, one run. The bracket on the coefficient is the spread over the row choices and the fit's error, not run to run or motor to motor.
- The brush path is a difference between two experiments on different days, and the rest-resistance sitting's temperature enters it through the coefficient. A degree of error in that temperature moves the copper by about 0.4 % and the brush path by the same amount of ohm.
- The solder on the tabs and the leads add a little mass to the rotor if it is weighed with them on, and the rotor's mass is to be taken before the leads go on.

## Open questions

- Whether the thermocouple in the coil reads the copper. A repeat of the plateaus with a drop of thermal paste under the tip and a second thermocouple on the rotor's iron, both logged, reads the gap between the two and how much a bare tip misses. A coefficient that moves toward copper's with the paste puts the difference on the tip.
- Whether the brush path is what the two routes read it as. The ring connection predicts the terminal reading at a bridging angle is three quarters of the two-segment reading less the brush path, and the twelve holes of rest-resistance have both angle kinds in them. The per-hole readings of that result against that prediction say whether the brush path is one number or moves with the angle.
- Spin-down without the field. The drag the no-load-current experiment reads is the bearings and the brushes together with the iron loss, the eddy and hysteresis loss the rotor's iron takes from turning in the magnets' field. With the magnet can off, the rotor turning in the end cap's bearing alone has no field and no iron loss, so a coast-down there is the friction on its own. The measurement is a tag on the pinion filmed at 240 fps, the rotor flicked up to speed by hand and left to coast, once in its can and once with the magnet arcs taken out of the can, the brushes lifted off the commutator both times so the bearings alone carry it, and the difference of the two decelerations at the same speed is the iron loss at that speed. No capture here, and the rotor has to go back into its can for it.
- The coefficient from motor to motor. A second teardown is not planned, one per family is the rule. If the stage 3 thermometer on the next motor of the family lands outside the accuracy this notebook gives it, that is the signal to open a second one.

## Conclusion

To be written from the capture. The highlight is the coefficient against copper's 0.00393 per C, and the share of the terminal resistance that is copper. If the coefficient lands within a few percent of copper's, the method reads the copper, and the stage 3 thermometer can carry copper's value with this notebook's bracket on it. The winding's $R_{20}$ and the brush path are what the in-situ experiments of stage 3 are graded against, and the counts go back into the registry entry.